# Requirements

In [125]:
import requests
from urllib.parse import quote
from IPython.display import JSON
from IPython.display import HTML, display
import html

# Inputs

In [4]:
API_SKETCHUP = "http://localhost:8000" 

# Functions

In [5]:
def get_skp_files() -> list:
    response = requests.get(f"{API_SKETCHUP}/files", timeout=15)
    response.raise_for_status()
    payload = response.json()
    return payload

In [6]:
def get_tree_from_skp_file(file_path: str) -> list:
    endpoint = f"{API_SKETCHUP}/tree"
    params = {"filepath": file_path}
    response = requests.get(endpoint, params=params, timeout=15)
    response.raise_for_status()
    payload = response.json()
    return payload

In [ ]:
def get_instances_from_skp_file(file_path: str) -> list:
    endpoint = f"{API_SKETCHUP}/instances"
    params = {"filepath": file_path}
    response = requests.get(endpoint, params=params, timeout=15)
    response.raise_for_status()
    payload = response.json()
    return payload

In [151]:
def set_instance_atribute_of_skp_file(file_path: str, guid: str, dictionary_name:str, key: str, value: str ) -> list:
    endpoint = f"{API_SKETCHUP}/update/instance"
    payload = {"file_path": file_path, 'guid': guid, 'dict_name': dictionary_name, 'key': key, 'value': value}
    response = requests.post(endpoint, json=payload, timeout=15)
    response.raise_for_status()
    payload = response.json()
    return payload

In [136]:

def render_tree(data, title="root"):
    # 1. Definimos o CSS uma única vez (usamos uma classe chamada 'json-key')
    css_style = """
    <style>
        .container {
            font-family: 'Segoe UI', sans-serif; 
            font-size: 15px!important; 

            margin-left: 5px;
            padding-left: 10px;

            line-height: 1.6; 
            list-style: none; 
            display: block;
            border-left: 1px solid rgba(87, 96, 111, 0.2);
        }
        .key {
            color: #57606f; 
        }
        .key_with_items {
            cursor: pointer;
        }
        .key_with_items:hover {
            opacity: 0.5 !important;
        }
        .value{
            color: #27ae60; 
            font-weight: 500; 
            margin-left: 5px;
        }
        details[open]>.key_with_items>.count {
            display: none;

        }

    </style>
    """
    def build_tree(curr_data, title):
        
        g_html = ""

        # CASO 1: DICIONÁRIO
        if isinstance(curr_data, dict):
            if len(curr_data.keys()) > 0:
                g_html += f"<details class='container'><summary class='key key_with_items'>{title} <b class='count'>{{{len(curr_data.keys())}}}</b></summary>"
                for key, value in curr_data.items():
                    g_html += build_tree(value, title=key)
                g_html += "</details>"
        
        elif isinstance(curr_data, list):
            if len(curr_data) > 0:
                g_html += f"<details class='container'><summary class='key key_with_items'>{title} <b class='count'>[{len(curr_data)}]</b></summary>"
                for i, item in enumerate(curr_data):
                    g_html += build_tree(item, title=f"#{i}")
                g_html += "</details>"
        
        else:
            g_html += f"<div class='container'><span class='key'>{title}:</span><span class='value'>'{html.escape(str(curr_data))}'</span></div>"
            
        return g_html
    
    

  # Retorna o CSS + o conteúdo da árvore
    g_html_output = css_style + build_tree(data, title)

    display(HTML(g_html_output))



# Analisys

In [11]:
skp_files = get_skp_files()

In [118]:
curr_file = skp_files[3]
curr_file_path = curr_file['path']

In [135]:
file_tree = get_tree_from_skp_file(curr_file_path)
render_tree(file_tree)

In [154]:
instances = get_instances_from_skp_file(curr_file_path)
render_tree(instances)

In [ ]:
set_instance_atribute_of_skp_file(curr_file_path, guid="2ENmTpu_v4RR$10TPnnfNQ", dictionary_name="dynamic_attributes", key="gbsid", value="1682214666666666")

{'success': True, 'detail': 'Atributo atualizado com sucesso na instancia.'}